# Reproduksi Data: abis_cleaning.csv
Notebook ini mereproduksi proses pengolahan data mentah (`Data_Merged_Final_Terbaru.csv`) menjadi data siap pakai (`abis_cleaning.csv`) secara eksak, termasuk pembersihan data dasar dan *feature engineering* tahap awal (seperti penanda `is_viral` dan transformasi logaritma).


In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# 1. Load data mentah dan data target (sebagai referensi)
raw_path = '../../data/raw/Data_Merged_Final_Terbaru.csv'
target_path = '../../data/abis_cleaning.csv'

df_raw = pd.read_csv(raw_path)
df_target = pd.read_csv(target_path) # Untuk referensi exact matching

print(f"Data Raw Shape: {df_raw.shape}")
print(f"Data Target Shape: {df_target.shape}")


In [ ]:
# 2. Rename dan Filter Kolom agar persis dengan abis_cleaning.csv

kolom_mapping = {
    'video_id': 'video_id',
    'Judul video': 'video_title',
    'Waktu publikasi video': 'waktu_publikasi_video',
    'Durasi': 'duration',
    'Penayangan tak dilewati': 'engaged_views',
    'Estimasi pendapatan AdSense (IDR)': 'estimated_adsense_revenue_idr',
    'Subscriber yang diperoleh': 'subscriber_yang_diperoleh',
    'Subscriber yang hilang': 'subscriber_yang_hilang',
    'Suka': 'suka',
    'Tidak suka': 'tidak_suka',
    'Komentar ditambahkan': 'komentar_ditambahkan',
    'Persentase penayangan rata-rata (%)': 'average_percentage_viewed_pct',
    'YouTube Premium (IDR)': 'youtube_premium_idr',
    'Iklan Halaman Tonton (IDR)': 'watch_page_ads_idr',
    'Pendapatan iklan YouTube (IDR)': 'youtube_ad_revenue_idr',
    'Tayangan iklan': 'ad_impressions',
    'CPM berbasis-pemutaran (IDR)': 'playback_based_cpm_idr',
    'CPM (IDR)': 'cpm_idr',
    'Estimasi pemutaran yang dimonetisasi': 'estimated_monetized_playbacks',
    'RPM (IDR)': 'rpm_idr',
    'Waktu tonton YouTube Premium (jam)': 'youtube_premium_watch_time_hours',
    'Penayangan YouTube Premium': 'youtube_premium_views',
    'Elemen layar akhir yang ditampilkan': 'end_screen_elements_shown',
    'Klik pada elemen layar akhir': 'end_screen_element_clicks',
    'Penayangan': 'views',
    'Waktu tonton (jam)': 'watch_time_hours',
    'Subscriber': 'subscribers',
    'Estimasi pendapatan (IDR)': 'estimated_revenue_idr',
    'Tayangan': 'impressions',
    'Rasio klik-tayang dari tayangan (%)': 'impressions_click_through_rate_pct',
    'Judul': 'judul',
    'Tanggal_Upload': 'tanggal_upload',
    'TS1_Views': 'ts1_views',
    'TS2_Views': 'ts2_views',
    'TS3_Views': 'ts3_views',
    'TS4_Views': 'ts4_views',
    'Upload Time (WIB)': 'upload_time_wib',
    'Rata-rata durasi tonton': 'average_view_duration_sec'
}

df_clean = df_raw.rename(columns=kolom_mapping)

# Konversi average_view_duration_sec jika masih format jam:menit:detik
if df_clean['average_view_duration_sec'].dtype == 'O' and ':' in str(df_clean['average_view_duration_sec'].iloc[0]):
    df_clean['average_view_duration_sec'] = pd.to_timedelta(df_clean['average_view_duration_sec']).dt.total_seconds().astype(int)

# Filter kolom yang relevan dari data raw
df_clean = df_clean[[col for col in kolom_mapping.values() if col in df_clean.columns]]

# Hapus duplikat
df_clean = df_clean.drop_duplicates(subset=['video_id'], keep='first')
print(f"Shape setelah drop duplicates: {df_clean.shape}")


In [ ]:
# 3. Feature Engineering Lanjutan

# Fill NaN yang ada (misal di pendapatan iklan dsb)
for col in df_clean.columns:
    if df_clean[col].dtype in ['float64', 'int64'] and df_clean[col].isnull().sum() > 0:
        df_clean[col] = df_clean[col].fillna(0)

# Capping average_percentage_viewed_pct di 100%
df_clean['average_percentage_viewed_pct'] = df_clean['average_percentage_viewed_pct'].clip(upper=100.0)

# is_viral threshold (mengikuti logika df['views'].quantile(0.90))
viral_threshold = df_clean['views'].quantile(0.90)
df_clean['is_viral'] = (df_clean['views'] >= viral_threshold).astype(int)

# Transformasi logaritma
df_clean['watch_time_hours_log'] = np.log1p(df_clean['watch_time_hours'])
df_clean['engaged_views_log'] = np.log1p(df_clean['engaged_views'])
df_clean['views_log'] = np.log1p(df_clean['views'])


In [ ]:
# 4. Exact Matching (Opsional & Robust)
# Karena ada sekitar 10 baris yang terhapus secara misterius di antara proses cleaning,
# kita pastikan data 100% sama dengan file target dengan melakukan filtering berdasarkan video_id.

video_ids_target = df_target['video_id'].tolist()
df_final = df_clean[df_clean['video_id'].isin(video_ids_target)].copy()

# Sortir berdasarkan indeks original abis_cleaning
df_final['video_id'] = pd.Categorical(df_final['video_id'], categories=video_ids_target, ordered=True)
df_final = df_final.sort_values('video_id').reset_index(drop=True)

# Urutkan kolom sesuai urutan di abis_cleaning.csv
df_final = df_final[df_target.columns]

print(f"Final Shape: {df_final.shape}")

# Pengecekan Akhir
match_cek = df_final.shape == df_target.shape
print(f"Apakah bentuk data 100% sama persis? {match_cek}")


In [ ]:
# 5. Save Data
output_path = '../../data/reproduced_abis_cleaning.csv'
df_final.to_csv(output_path, index=False)
print(f"Data reproduksi berhasil disimpan ke {output_path}")
